# Retrieval

LangChain에서 Retrieval은 외부 데이터에서 관련 정보를 찾아 프롬프트에 포함시켜(Context) LLM에 전달하는 역할을 한다. 주요 구성 요소는 다음과 같다.

- **Document Loader**: 다양한 원본 데이터를 LangChain 표준 문서 객체로 변환한다.
- **Text Splitter**: 긴 문서를 작은 청크로 분할해 검색 효율을 높인다.
- **Embedding Model**: 텍스트를 의미 기반 벡터로 변환한다.
- **Vector Store**: 임베딩된 벡터를 저장하고 유사도 기반 검색을 지원한다.
- **Retriever**: 쿼리에 대해 관련 문서를 찾아주는 표준 인터페이스를 제공한다.
- **Retrieval Chain**: 검색된 문서를 LLM에 전달해 답변을 생성하는 체인 구조를 제공한다.

이렇게 각 모듈이 결합되어, 외부 데이터 기반의 효과적인 검색 및 답변 생성이 가능하다.

**환각 Hallucination:**

LLM이 실제 근거 없이 그럴듯해 보이는 정보를 생성하는 현상이다.

**주요 원인**
1. **학습 데이터 한계**
   * 모델이 학습한 데이터에 해당 정보가 없거나 부족할 때 발생한다.
2. **확률적 생성 과정**
   * 토큰 예측 시 언어적 일관성을 우선하다 보니, 사실 여부가 검증되지 않은 내용을 생성한다.
3. **프롬프트 모호성**
   * 지시가 불명확하거나 맥락이 부족하면 모델이 관련 없는 정보를 보충·왜곡한다.

**대표 사례**
* 존재하지 않는 논문·저자명을 인용함.
* 역사적·과학적 사실을 잘못 기술함.
* 실행 불가능하거나 비효율적인 코드 제안.


**완화 방안**

1. **지식 기반 검색 결합**
   * Retrieval-Augmented Generation(RAG) 방식으로 외부 문서·데이터베이스에서 실시간 근거를 가져와 보강한다.
2. **프롬프트 구체화**
   * “출처를 함께 제시해 달라” 등 명시적 요청을 통해 근거 표기를 유도한다.
3. **후처리 검증**
   * 생성 결과를 룰 기반 검증 또는 전문가 리뷰를 통해 교차 확인한다.
4. **모델 파인튜닝 및 앙상블**
   * 도메인 특화 데이터로 추가 학습하거나, 룰 기반 시스템과 결합하여 정확도를 높인다.

In [1]:
#%pip install langchain-community langchain-openai langchain-huggingface wikipedia pypdf tavily-python tiktoken faiss-cpu

In [2]:
%pip install langchain-community langchain-openai langchain-huggingface wikipedia pypdf tavily-python tiktoken faiss-cpu sentence-transformers -Uqqq

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
# os.environ['LANGSMITH_ENDPOINT'] = 'https://apac.api.smith.langchain.com'
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

## Document

Document는 LangChain 프레임워크에서 다양한 데이터 소스(예: 텍스트 파일, PDF, 웹페이지 등)로부터 불러온 정보를 표준화된 객체로 표현하는 핵심 데이터 구조이다. 이 객체는 언어 모델(LLM)이 외부 데이터를 이해하고 처리할 수 있도록 도와준다.

**Document 객체의 구조**
1. page_content: 문서의 실제 내용을 담고 있는 문자열(str)이다. 예를 들어, 텍스트 파일의 본문이나 PDF의 텍스트 등이 여기에 저장된다.
2. metadata: 문서에 대한 부가 정보를 담는 딕셔너리(dict) 형태의 속성이다. 예를 들어, 파일 경로, 페이지 번호, 작성자, 데이터 출처 등 다양한 메타데이터를 저장할 수 있다.


**Document의 역할과 활용**
1. 표준화된 데이터 구조: 다양한 포맷의 데이터를 일관된 방식으로 표현하여, LLM이 손쉽게 접근하고 활용할 수 있도록 한다.
2. 문서 처리의 기본 단위: LangChain의 문서 로더(Document Loader)는 파일, 웹, 데이터베이스 등 여러 소스에서 데이터를 읽어와 Document 객체로 변환한다.
3. 청크 단위 분할: 대용량 문서는 작은 단위(청크)로 쪼개어 각각의 Document로 저장하고, 검색 및 임베딩 처리에 활용한다.

In [4]:
from langchain_core.documents import Document

doc = Document(
    # 문서 본문 텍스트
    page_content= '이것은 랭체인의 Document 객체입니다. 모든 데이터소스는 이 Document 객체로 변환됩니다.',
    # 문서에 붙는 부가정보
    metadta = {
        'source':'여기저기',             # 데이터 출처
        'url':'https://encore.com',     # 원문 URL
        'timestamp': 2020608250939      # 수집/생성 시간
    }
)

print(doc)
print(doc.page_content)
print(doc.metadata)


page_content='이것은 랭체인의 Document 객체입니다. 모든 데이터소스는 이 Document 객체로 변환됩니다.'
이것은 랭체인의 Document 객체입니다. 모든 데이터소스는 이 Document 객체로 변환됩니다.
{}


## Document Loader
https://reference.langchain.com/python/langchain_core/document_loaders/


Document Loader는 다양한 데이터 소스에서 데이터를 읽어와 Document 객체로 변환하는 역할을 한다. 예를 들어, PDFLoader, CSVLoader, TextLoader 등 다양한 종류가 존재하며, 각기 다른 파일 형식을 Document 객체로 표준화한다.

Document Loader는 데이터 소스별로 특화된 클래스를 제공하며, 문서를 로드한 후 LangChain에서 사용하는 표준 형식으로 변환해준다.

1. **다양한 데이터 소스 지원**  
   Document Loader는 파일 시스템, 클라우드 스토리지, 데이터베이스, 웹 등 다양한 데이터 소스에서 데이터를 로드할 수 있도록 설계되었다.
   
2. **표준화된 출력 형식**  
   로드된 문서는 LangChain에서 사용하는 `Document` 객체로 변환된다. `Document` 객체는 다음과 같은 필드를 포함한다:
   - `page_content`: 문서 본문 내용
   - `metadata`: 문서와 관련된 메타데이터 (예: 파일 이름, URL, 작성자 등)

3. **플러그인 기반 확장 가능**  
   사용자 정의 데이터 소스 로더를 쉽게 구현하고 LangChain에 통합할 수 있다.

**주요 Document Loader 예시**

| Loader 이름        | 설명                                                              |
|--------------------|-------------------------------------------------------------------|
| `PyPDFLoader`      | PDF 문서를 로드하며 텍스트를 추출해 Document 형식으로 변환한다.     |
| `TextLoader`       | 일반 텍스트 파일을 로드한다.                                      |
| `UnstructuredFileLoader` | 비구조적 데이터를 로드하여 구조화된 텍스트로 변환한다.           |
| `CSVLoader`        | CSV 파일에서 데이터를 로드하며 행(row)을 Document로 처리한다.      |
| `WebBaseLoader`    | 웹 페이지 데이터를 크롤링하여 Document로 로드한다.                |

In [5]:
from langchain_community.document_loaders import WebBaseLoader

url = 'https://n.news.naver.com/article/047/0002526580'


header = {
    # 브라우저 식별 : Windows에서 Chrome 으로 접속한 것 처럼 보이게 만드는 UA
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

loader = WebBaseLoader(url, header_template=header) # 로더 객체 생성
docs = loader.load() # 불러온 웹페이지 -> Document 리스트
docs

C:\Users\playdata2\AppData\Local\Temp\ipykernel_18500\2564720278.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://n.news.naver.com/article/047/0002526580', 'title': "'두 번 멸종한 동물'의 정체... 인간은 참 오만했다", 'language': 'ko'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\'두 번 멸종한 동물\'의 정체... 인간은 참 오만했다\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n본문 바로가기\n\n\n\n\n\n\n이전 페이지\n\n\n\n\n\n\n\n\n\n\n오마이뉴스\n\n\n\n\n\n구독\n\n메인 뉴스판에서 오마이뉴스 주요뉴스를 볼 수 있습니다.\n보러가기\n닫기\n\n\n오마이뉴스 언론사 구독 해지되었습니다.\n닫기\n\n\n\n\n\n\n\n\n\n\n주요뉴스\n클립\n이슈\n정치\n경제\n사회\n생활\n세계\n랭킹\n\n\n\nMY\n\n\n뉴스 이용 설정을 할 수 있어요\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n오마이뉴스\n\n\n\nPICK\n안내\n\n\n언론사가 주요기사로선정한 기사입니다.\n언론사별 바로가기\n닫기\n\n\n\n\n\'두 번 멸종한 동물\'의 정체... 인간은 참 오만했다\n\n\n\n\n\n\n\n\n입력\n2026.08.25. 오전 7:26\n\n\n\n기사원문\n \n\n\n\n\n\n\n\n\n\n\n추천\n반응\n\n\n\n\n쏠쏠정보\n0\n\n\n\n\n흥미진진\n0\n\n\n\n\n공감백배\n0\n\n\n\n\n분석탁월\n0\n\n\n\n\n후속강추\n0\n\n\n \n\n\n\n\n댓글\n반응\n\n\n\n\n\n\n\n\n텍스트 음성 변환 서비스 사용하기\n\n\n\n성별\n남성\n여성\n\n\n말하기 속도\n느림\n보통\n빠름\n\n이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다.\n본문듣기 시작\n\n닫기\n\n\n \n\n글자 크기 변경하기\n\n글자크

In [6]:
print(len(docs))
doc = docs[0]
print(doc.metadata) # 메타데이터
print(doc.metadata['title']) # 메타데이터 title
print(doc.page_content.replace('\n','')) # 본문 내용

1
{'source': 'https://n.news.naver.com/article/047/0002526580', 'title': "'두 번 멸종한 동물'의 정체... 인간은 참 오만했다", 'language': 'ko'}
'두 번 멸종한 동물'의 정체... 인간은 참 오만했다
'두 번 멸종한 동물'의 정체... 인간은 참 오만했다본문 바로가기이전 페이지오마이뉴스구독메인 뉴스판에서 오마이뉴스 주요뉴스를 볼 수 있습니다.보러가기닫기오마이뉴스 언론사 구독 해지되었습니다.닫기주요뉴스클립이슈정치경제사회생활세계랭킹MY뉴스 이용 설정을 할 수 있어요오마이뉴스PICK안내언론사가 주요기사로선정한 기사입니다.언론사별 바로가기닫기'두 번 멸종한 동물'의 정체... 인간은 참 오만했다입력2026.08.25. 오전 7:26기사원문 추천반응쏠쏠정보0흥미진진0공감백배0분석탁월0후속강추0 댓글반응텍스트 음성 변환 서비스 사용하기성별남성여성말하기 속도느림보통빠름이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다.본문듣기 시작닫기 글자 크기 변경하기글자크기가1단계작게가2단계보통가3단계크게가4단계아주크게가5단계최대크게닫기SNS 보내기인쇄하기[이하늬의 멸종위기동물] 피레네아이백스의 잔혹사황량하고 혹독한 피레네산맥의 깎아지른 절벽, 인간의 발길이 쉽게 닿지 않는 협곡 사이를 바람처럼 누비던 동물이 있었습니다. 65~90cm에 달하는 웅장하고 아름다운 뿔, 험준한 바위 산책로를 마치 평지처럼 유희하듯 거닐던 우아한 자태의 소유자였습니다.피레네아이벡스(Pyrenean ibex), 현지에서는 '부카도(Bucardo)'라 불리던 이베리아아이벡스의 아종입니다. 그러나 지금 피레네산맥 어디에서도 진짜 피레네아이벡스의 실루엣은 찾아볼 수 없습니다. 이들은 지구상에서 유일하게 '두 번 멸종한 동물'이라는 서글프고도 기이한 기록을 남긴 채 역사 속으로 사라졌습니다.기술이 모든 것을 해결해 줄 것이라는 21세기의 오만, 그리고 한때 지구를 함께 공유했던 야생 생명에 대한 뒤늦은 사과

In [7]:
# 톰소여의 여행(영문)
!gdown 1o7ngiyeJJ-MPLhl0fiCKHViTNNpk6zjO

Downloading...
From: https://drive.google.com/uc?id=1o7ngiyeJJ-MPLhl0fiCKHViTNNpk6zjO
To: c:\Users\playdata2\LLM\05_langchain\02_langchain_component\The_Adventures_of_Tom_Sawyer.pdf

  0%|          | 0.00/2.68M [00:00<?, ?B/s]
 20%|█▉        | 524k/2.68M [00:00<00:00, 3.13MB/s]
 78%|███████▊  | 2.10M/2.68M [00:00<00:00, 7.98MB/s]
100%|██████████| 2.68M/2.68M [00:00<00:00, 8.44MB/s]


In [8]:
from langchain_community.document_loaders import PyPDFLoader # PDF 파일을 읽어 Document로 로드하는 로더

loader = PyPDFLoader('The_Adventures_of_Tom_Sawyer.pdf')
docs = loader.load() # PDF를 페이지별 Document 리스트로 변환
print(len(docs)) # Documnet 수


35


In [9]:
print(docs[2].metadata) # 3 페이지의 메타데이터
print(docs[2].metadata['source']) # 경로
print(docs[2].metadata['page']) # 현재 페이지
print(docs[2].metadata['page_label']) # 사람이 볼 수있는 페이지 번호
print(docs[2].page_content)  # 본

{'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 2, 'page_label': '3'}
The_Adventures_of_Tom_Sawyer.pdf
2
3
The Adventures of                 
Tom Sawyer 
 
MARK TWAIN 
Level 1 
 
Retold by Jacqueline Kehl                                                    
Series Editors: Andy Hopkins and Jocelyn Potter


### TavilySearchAPIRetriever
https://www.tavily.com/

- `langchain_tavily.TavilySearch`: Agent tool사용버젼. json반환
- `langchain_community.retrievers.TavilySearchAPIRetriever`: 검색기(context확보용) Document객체반환

- 주요 기능
    - 웹 검색(query → 결과 리스트): 키워드로 웹을 검색해서 관련 페이지들을 찾아줌
    - 요약/스니펫 제공: 각 결과에 본문 요약이나 핵심 스니펫을 같이 줘서 LLM이 바로 쓰기 좋음
    - 컨텐츠 추출(include_raw_content 등 옵션): 결과 페이지의 내용을 일부/전체 텍스트로 가져오게 설정 가능
    - 필터링/튜닝 옵션: 검색 결과 개수, 도메인 포함/제외, 최신성(리센시) 같은 옵션으로 결과를 조절 가능
    - RAG 파이프라인에 바로 연결: “검색 → 문서(Document)화 → 벡터화/리랭킹 → 답변” 흐름에서 검색 단계로 많이 사용

In [10]:
%pip install tavily-python -qqq

Note: you may need to restart the kernel to use updated packages.


In [11]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [12]:
from langchain_community.retrievers import TavilySearchAPIRetriever

tavily_retriever = TavilySearchAPIRetriever(k=3) # 검색 결과 상위 3개

docs = tavily_retriever.invoke("런닝") # 검색어로 웹 검색 -> Document 리스트
docs

[Document(metadata={'title': "'러닝', '런닝' 중 올바른 표기는? (런닝머신? 러닝머신!) : 네이버 블로그", 'source': 'https://m.blog.naver.com/chorduk/223308645264', 'score': 0.7395922, 'id': 'fc79ad-00', 'images': []}, page_content="우리말샘\n\n\u200b\n\n「001」 「명사」 운동 경기할 때 선수들이 입는 소매 없는 셔츠. 또는 그런 모양의 속옷.\n\n\u200b\n\n「002」 「명사」 『체육』 요트 경기에서, 바람을 등지고 달리는 일.\n\n\u200b\n\n「003」 「명사」 『체육』 스키에서, 미끄러져 내려가는 일.\n\n\u200b\n\n「004」 「명사」 『체육』 외국 프로 레슬링에서, 상대를 잡아 들어 올린 후 앞으로 뛰어가 기둥이나 물체에 상대를 부딪치게 하는 기술.\n\n\u200b\n\n이렇게 나와 있습니다.\n\n\u200b\n\n\u200b\n\n\u200b\n\n음.. 그럼 '런닝'은 뭘까요?\n\n\u200b\n\n'러닝'과 같거나 비슷한 뜻을 가진 단어일까요?\n\n\u200b\n\n'러닝'과 다른 뜻을 가진 단어일까요?\n\n\u200b\n\n아니면 '러닝'을 잘못 표기한 걸까요? \u200b\n\n\u200b\n\n\n\n\u200b\n\n\u200b\n\n\u200b\n\n사전에 '런닝'도 검색해 보도록 하겠습니다.\n\n\u200b\n\n\u200b\n\n\u200b\n\n우리말샘에 '런닝'을 검색해 보면\n\n\n\n우리말샘\n\n\u200b [...] 우리말샘\n\n\u200b\n\n「001」 「명사」 운동 경기할 때 선수들이 입는 소매 없는 셔츠. 또는 그런 모양의 속옷. ⇒규범 표기는 ‘러닝’이다.\n\n\u200b\n\n「002」 「명사」 「북한어」 ‘러닝’의 북한어.\n\n\u200b\n\n「003」 「명사」 『체육』 외국 프로 레슬링에서, 상대를 잡아 들어 올린 후 앞으로 

In [13]:
for doc in docs:
    print(doc.page_content)

우리말샘

​

「001」 「명사」 운동 경기할 때 선수들이 입는 소매 없는 셔츠. 또는 그런 모양의 속옷.

​

「002」 「명사」 『체육』 요트 경기에서, 바람을 등지고 달리는 일.

​

「003」 「명사」 『체육』 스키에서, 미끄러져 내려가는 일.

​

「004」 「명사」 『체육』 외국 프로 레슬링에서, 상대를 잡아 들어 올린 후 앞으로 뛰어가 기둥이나 물체에 상대를 부딪치게 하는 기술.

​

이렇게 나와 있습니다.

​

​

​

음.. 그럼 '런닝'은 뭘까요?

​

'러닝'과 같거나 비슷한 뜻을 가진 단어일까요?

​

'러닝'과 다른 뜻을 가진 단어일까요?

​

아니면 '러닝'을 잘못 표기한 걸까요? ​

​



​

​

​

사전에 '런닝'도 검색해 보도록 하겠습니다.

​

​

​

우리말샘에 '런닝'을 검색해 보면



우리말샘

​ [...] 우리말샘

​

「001」 「명사」 운동 경기할 때 선수들이 입는 소매 없는 셔츠. 또는 그런 모양의 속옷. ⇒규범 표기는 ‘러닝’이다.

​

「002」 「명사」 「북한어」 ‘러닝’의 북한어.

​

「003」 「명사」 『체육』 외국 프로 레슬링에서, 상대를 잡아 들어 올린 후 앞으로 뛰어가 기둥이나 물체에 상대를 부딪치게 하는 기술. ⇒규범 표기는 ‘러닝’이다.

​

보시다시피 이렇게 나와 있습니다.

​

​

​

음.. 그럼 왜 '런닝'이 아닌 '러닝'일까요? ​

​



​

​

​

이건 '러닝(running)'의 발음 기호를 보면 알 수 있습니다.

​

'러닝(running)'의 발음 기호는 [ˈrʌnɪŋ]입니다.

​

'n'이 하나만 있죠?

​

따라서 '런닝'이 아닌 '러닝'으로 적는 것이 올바른 표기입니다. ​

​



​

​

​

그럼 이만 포스팅 마치도록 하겠습니당😆😆

​

​

​

끝까지 읽어 주셔서 감사합니다🧡​

​



​ [...] 본문 바로가기

# 블로그

## 카테고리 이동 고양이 좋아하는 속기사

검색

'러닝', '런닝' 중 

In [14]:
docs[2].page_content

'기본적으로 러닝은 관절이나 인대 및 지근 등이 강화되어야 페이스를 단축하거나, 인터벌 훈련을 할 밑바탕이 깔리기 때문에 초보자라면 무리한 속도를 내는 것을 지양하고 거리와 시간을 목표로 잡아야 한다. 최소 10km를 1시간 동안(페이스 6min/km) 연속으로 달릴 수 있는 체력이 되고 나서야 페이스 단축을 시도하자. 이조차 안되면서 빨리 달리는 것을 먼저 해내려는 것은 부상으로 가는 지름길이다. [...] 최근 변경최근 토론\n\n특수 기능\n\n\n\n\n\n "달리기(노땐스)") 문서를, 옥상달빛의 노래에 대한 내용은 달리기(옥상달빛) "달리기(옥상달빛)") 문서를, QWER의 노래에 대한 내용은 달리기(QWER) "달리기(QWER)") 문서를 참고하십시오.\n\nImage 5Image 6: 다른 뜻 아이콘러닝은 여기로 연결됩니다. 민소매 속옷을 가리키는 런닝에 대한 내용은 러닝셔츠 문서를 참고하십시오.\n\nImage 8Image 9: 육상 픽토그램올림픽 육상의 세부 종목\n트랙필드복합도로\n달리기\n\n(100m)멀리뛰기포환던지기10종/7종경보 "경보(스포츠)")\n허들\n\n(100m(여)/110m(남) · 400m)세단뛰기원반던지기\n높이뛰기창던지기마라톤\n계주장대높이뛰기해머던지기\n\nImage 11Image 12: attachment/Runni...\n\n1. 개요2. 인간의 달리기 특성3. 달리기와 걷기의 차이점4. 달리기와 걷기에 대한 낭설5. 달리기의 방법\n\n5.1. 착지법5.2. 호흡법'

### Tavily 검색 결과로 Context에 넣고 답변하는 RAG Chain

In [15]:
from langchain_core.prompts import PromptTemplate # 프롬프트 템플릿 생성
from langchain.chat_models import init_chat_model # 랭체인 ChatModel 구성
from langchain_core.output_parsers import StrOutputParser # LLM 응답 -> 문자열
from langchain_core.runnables import RunnablePassthrough # 입력을 그대로 통과  시키는 Runnable
from langchain_core.documents import Document # Document 객체

tavily_retriever = TavilySearchAPIRetriever(k=3) # 검색 결과 상위 3개
prompt = PromptTemplate.from_template('''
    사용자의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.
    Context : {context}
    Question : {question}
''')

llm = init_chat_model('openai:gpt-5.6-luna')
output_parser = StrOutputParser() # 최종 출력은 텍스트 형식

# 검색된 Document 리스트를 프롬프트에 넣기 좋은 문자열 형태로 합치는 함수 
def format_docs(docs: list[Document]) -> str:
    # 문서 본문끼리 개행문자로 이어붙인 문자열
    return '\n\n'.join(doc.page_content for doc in docs) 


tavily_chain = tavily_retriever | format_docs # (질문) -> 검색 -> 텍스트

# question 은 입력값 그대로 전달, context는 tavily 검색 결과 | 최종 프롬프트 완성 | LLM 호출 | 문자열 파싱
chain = (
    {'question' : RunnablePassthrough(), 'context': tavily_chain} | prompt | llm | output_parser
)

print(chain.invoke('8월 말 독산역 인기 맛집?'))

8월 말 독산역 인근에서 인기와 방문 목적을 기준으로 고르면 다음 맛집을 추천합니다. 다만 8월 말 특별 행사나 계절 메뉴 정보는 확인되지 않습니다.

1. **화로구이 독산점**
   - 삼겹살, 돼지갈비, 한우를 먹기 좋은 독산동 대표 고기집
   - 블로그 리뷰 434개, 방문자 리뷰 1,280개로 이용자 평가가 많은 편
   - 샐러드바와 냉면이 있어 여름 외식이나 단체 회식에 적합
   - 독산역 1번 출구 건너편에서 약 200m
   - **24시간 연중무휴**

2. **진영면옥**
   - 8월 말 더운 날씨에 평양냉면, 비빔냉면을 찾는다면 추천
   - 곰탕, 수육, 녹두전도 함께 판매
   - 조용하고 정갈한 분위기로 혼자 방문하기에도 괜찮은 곳
   - 평일·토요일 11:00~21:00, 15:00~17:00 브레이크타임
   - 일요일 휴무

3. **기주짬뽕 본점**
   - 짬뽕, 해물짬뽕, 간짜장, 탕수육을 찾는 사람에게 적합
   - 블로그 리뷰 65개, 방문자 리뷰 282개
   - 홈플러스 뒤편 현대지식산업센터 B동 1층에 위치
   - 영업시간은 제공된 정보에 없어 방문 전 확인이 필요합니다.

4. **궁전산들애**
   - 한정식, 간장게장, 옥돔구이, 불고기전골 등 한식 메뉴 중심
   - 가족 외식이나 특별한 날, 단체 모임에 적합
   - 블로그 리뷰 277개, 방문자 리뷰 974개
   - 매일 11:00~23:00

5. **팔곱집**
   - 곱창·막창·양과 술자리를 즐기기 좋은 독산동 식당
   - 회식이나 저녁 모임에 어울림
   - 독산역 인근 독산1동주민센터와 롯데빅마켓 주차장 부근
   - 정확한 영업시간은 확인이 필요합니다.

6. **우마왕**
   - 소고기 특수부위와 삼합을 찾을 때 고려할 만한 곳
   - 독산역 1번 출구에서 약 9분 거리
   - 고기류를 선호하면서 화로구이와 다른 메뉴를 찾는 경우 적합

**한 곳만 고른다면**
- 더운 날 냉면: **진영면옥**
- 고기·회식: **화로구

## Embedding Model
- openai
- setence-transformer(huggingface)

In [16]:
from langchain_openai import OpenAIEmbeddings
import pandas as pd

# 임베딩 모델(1536차원)
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

text = '철수는 골든리트리버를 키우고 있습니다.'

emb_vec = embeddings.embed_query(text) # 텍스트 임베딩 -> float 리스트(벡터) 변
print(len(emb_vec)) # 임베딩 차원 
print(emb_vec[:3]) # 샘플 3개 확인

pd.Series(emb_vec, name="embedding")



1536
[-0.0273590087890625, -0.00527191162109375, 0.005352020263671875]


0      -0.027359
1      -0.005272
2       0.005352
3      -0.049652
4       0.026611
          ...   
1531    0.022980
1532   -0.026749
1533   -0.001171
1534    0.036957
1535   -0.023468
Name: embedding, Length: 1536, dtype: float64

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

# 임베딩 모델 (384차원)
embeddings = HuggingFaceEmbeddings(model = 'sentence-transformers/all-MiniLM-L6-v2')

text = '철수는 골든리트리버를 키우고 있습니다.'

emb_vec = embeddings.embed_query(text) # 텍스트 임베딩 -> float 리스트(벡터) 변
print(len(emb_vec)) # 임베딩 차원 
print(emb_vec[:3]) # 샘플 3개 확인

pd.Series(emb_vec, name="embedding")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

384
[0.011021507903933525, 0.06504975259304047, 0.048188380897045135]


0      0.011022
1      0.065050
2      0.048188
3     -0.068704
4      0.014492
         ...   
379    0.067011
380    0.011610
381    0.025551
382   -0.018868
383    0.005621
Name: embedding, Length: 384, dtype: float64

## Vector Store

벡터 데이터베이스란 쉽게 말해, **비정형 데이터(텍스트, 이미지, 오디오 등)를 숫자 벡터로 변환하여 저장하고, 이 벡터들 간의 유사성을 바탕으로 데이터를 검색**하는 데이터베이스를 말한다. 여기서 벡터는 데이터를 다차원 공간에서 표현한 수학적 객체이다.

- **벡터**: 데이터의 특징을 다차원으로 표현한 값.
  - 예: 단어 임베딩은 단어를 벡터로 변환하여 유사한 단어들이 가까이 위치.
- **벡터 데이터베이스 필요성**:
  - RDBMS는 구조화된 데이터(테이블 형태)에 적합.
  - AI/머신러닝의 발전으로 비정형 데이터를 처리할 필요 증가.
  - 벡터 데이터베이스는 **유사도 기반 검색**으로 고차원 데이터 처리에 유리.

**주요 특징:**
- 유사한 데이터를 빠르게 검색.
- AI 응용 분야(이미지 검색, 자연어 처리, 추천 시스템 등)에서 중요.

- **벡터 데이터베이스와 RDBMS의 주요 차이점**

| **특징**                | **RDBMS**                                                                 | **벡터 데이터베이스**                                                                                  |
|-------------------------|---------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------|
| **데이터 구조**          | 테이블 형식으로 데이터 저장, SQL을 사용하여 질의.                             | 다차원 벡터 형식으로 데이터 저장, 벡터 간 유사도 계산 방식 사용.                                        |
| **검색 방식**            | 키-값 쌍이나 고정 조건 기반 검색 (정확한 일치 검색).                          | 유사성 검색 수행, 벡터 간 거리(예: 코사인 유사도, 유클리드 거리)를 기준으로 유사한 데이터를 반환.         |
| **비정형 데이터 처리**   | 텍스트, 숫자 등 구조화된 데이터 처리에 적합.                                 | 이미지, 오디오, 영상 등 비정형 데이터를 벡터로 변환해 처리 가능.                                       |
| **응용 분야**            | 전통적인 CRUD 작업, 금융 데이터, 고객 데이터 관리 등.                       | AI 기반 추천 시스템, 이미지 검색, 자연어 처리, 음성 인식 등.                                           |
| **확장성**               | 수평 확장 가능하지만 고차원 데이터나 복잡한 쿼리 처리에는 한계.               | 수백만~수십억 개 벡터 데이터를 효율적으로 처리 가능.                                                  |

**벡터 데이터베이스의 주요 특징**

1. **Approximate Nearest Neighbor (ANN) 검색**  
   - **ANN 알고리즘**을 사용해 유사한 벡터를 빠르게 검색.  
   - 검색 속도가 빠르고, 대규모 데이터셋에서도 효율적으로 동작.

2. **확장성**  
   - 수백만~수십억 개의 벡터 데이터를 처리할 수 있는 구조로 설계.  
   - 대규모 데이터셋에서 고속 검색 및 처리가 가능.

3. **유연성**  
   - 텍스트, 이미지, 오디오 데이터를 임베딩 형태로 변환해 저장 가능.  
   - 다양한 머신러닝 모델과 통합하여 사용자 요구에 맞는 검색 시스템 구축 가능.

**주요 벡터 데이터베이스 비교**

| **이름**      | **특징**                                                                                                                                   | **장점**                                                                                                             | **단점**                                                                                  |
|---------------|-------------------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------|
| **Chroma**    | 오픈 소스 벡터 데이터베이스, LLM(대규모 언어 모델) 응용에 최적화. Python 노트북 환경에서 간편하게 사용 가능하며 프로덕션으로 확장 가능.                | 간편한 설정, 유사성 검색 및 임베딩 관리 용이, LLM 응용 프로그램에 적합.                                               | 대규모 데이터 처리에서 다른 서비스만큼 최적화되어 있지 않을 수 있음.                                          |
| **Pinecone**  | 완전 관리형 서비스로, 대규모 고차원 데이터의 실시간 처리 및 검색에 최적화.                                                                  | 유지보수 불필요(관리형 서비스), 실시간 대규모 데이터 검색에 강점, 데이터 엔지니어 및 과학자들에게 적합.                 | 오픈 소스가 아니며, 서비스 사용 비용이 발생.                                                              |
| **Weaviate**  | 오픈 소스 기반, OpenAI, Cohere, HuggingFace와의 통합으로 벡터화 작업 용이.                                                                 | 다양한 플랫폼과의 통합 기능, 확장성과 유연성, 고차원 데이터 검색 성능 우수.                                           | 복잡한 설정 및 사용 시 초기 학습 필요.                                                              |
| **Faiss**     | Meta에서 개발한 라이브러리로 대규모 벡터 세트 검색에 최적화. Python 및 GPU 지원으로 성능 극대화.                                              | 고성능 검색(GPU 지원), 대규모 데이터셋 처리 능력, 빠른 속도.                                                          | 데이터베이스가 아닌 라이브러리 형태로 제공되어, 추가적인 환경 설정 및 통합 작업 필요.                                         |
| **Qdrant**    | Rust로 구현된 API 기반 벡터 검색 도구. 빠른 검색과 자원 최적화를 제공하며 정교한 필터링 기능 지원.                                             | 뛰어난 성능(Rust 기반), 정교한 필터링 기능, API 중심의 유연한 설계.                                                    | 커뮤니티와 생태계가 다른 데이터베이스에 비해 상대적으로 작을 수 있음.                                         |

**선택 가이드**
1. **LLM 응용 프로그램**: Chroma, Weaviate.  
2. **완전 관리형 서비스**: Pinecone.  
3. **고성능 및 GPU 지원 필요**: Faiss.  
4. **정교한 필터링과 최적화된 성능**: Qdrant.  

### FAISS

- **공식 문서**: https://faiss.ai/
- **Github**: https://github.com/facebookresearch/faiss

**Faiss(Vector Search Library)**는 Facebook AI Research에서 개발한 **효율적인 벡터 검색 및 밀집 벡터 인덱싱 라이브러리**이다. 대규모 데이터에서 **빠른 유사도 검색과 군집화**를 수행하는 데 최적화되어 있다. 주로 문서 검색, 추천 시스템, 이미지 검색, NLP 모델에서 벡터 임베딩 처리를 지원한다.

**주요 특징**
1. **효율적인 유사도 검색**
   - `k-NN (k-Nearest Neighbors)`를 기반으로 벡터 간 유사도(예: 코사인 유사도, L2 거리)를 계산한다.
   - CPU/GPU 모두 지원하여 대규모 데이터에서도 빠르게 처리 가능하다.

2. **고성능 인덱싱**
   - 다양한 **인덱싱 알고리즘**(Flat, IVF, HNSW, PQ 등)을 지원하여 정확도와 속도 간 균형을 맞출 수 있다.
   - 데이터가 커질수록 효율적으로 검색 성능을 발휘하도록 설계되었다.

3. **확장성**
   - 수억 개의 벡터에서도 성능을 유지하도록 설계되었으며, GPU 병렬 처리를 통해 성능을 극대화한다.

4. **유연성**
   - Python과 C++ API를 제공하며, Scikit-learn이나 PyTorch와 같은 다른 라이브러리와 통합하여 사용 가능하다.

**Faiss의 기본 인덱스 유형**
1. **Flat Index**
   - 모든 벡터를 저장하고 전체 탐색(Brute-Force)을 수행.
   - 정확도가 높지만 대규모 데이터에서는 속도가 느릴 수 있다.

2. **IVF (Inverted File Index)**
   - 벡터를 클러스터링하여 데이터 양을 줄이고 탐색 속도를 높임.
   - 대규모 데이터에서 적합하며, 정확도와 속도 조절 가능.

3. **PQ (Product Quantization)**
   - 벡터를 압축하여 메모리 사용량을 줄이고, 빠른 근사 유사도 검색 수행.

4. **HNSW (Hierarchical Navigable Small World Graphs)**
   - 그래프 기반 알고리즘으로 매우 빠른 근사 유사도 검색 가능.


**Faiss의 주요 사용 사례**
1. **문서 검색**
   - 문서를 벡터로 변환한 후 가장 관련 있는 문서를 검색.
   - NLP 모델의 임베딩과 결합하여 사용.

2. **이미지 검색**
   - 이미지 특징 벡터를 사용하여 비슷한 이미지를 검색.

3. **추천 시스템**
   - 사용자의 행동이나 관심사를 벡터화하여 추천 품목 생성.

4. **클러스터링**
   - 벡터 데이터를 군집화하여 데이터의 구조를 분석.

In [18]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
import numpy as np

loader = PyPDFLoader('The_Adventures_of_Tom_Sawyer.pdf') # PDF 로더 객체 생성
docs = loader.load() # PDF -> 페이지 단위 Document
page_contents = [doc.page_content for doc in docs] # 각 페이지의 텍스트를 리스트로 추출

embeddings = OpenAIEmbeddings(model= 'text-embedding-3-small') # 1536 차원
emb_vecs = embeddings.embed_documents(page_contents) # 페이지별 컨텐츠 임베딩 -> 벡터 리스트
np.array(emb_vecs).shape #(페이지수, 임베딩 차원수)

(35, 1536)

In [19]:
# FAISS 벡터스토어를 이용해 문서들을 임베딩 해 로컬에 저장
from langchain_community.vectorstores import FAISS # FAISS 기반 벡터 DB(Vector Store)

vector_db = FAISS.from_documents(docs, embeddings) # docs 를 임베딩해서 FAISS 인덱스생성
vector_db.save_local('./db/faiss') # 로컬 경로에 FAISS 인덱스/메타데이터 저장

In [20]:
# 로컬에 저장해놓은 FAISS 벡터스토어 로드
vector_db = FAISS.load_local(
    './db/faiss', # 경로
    embeddings, # 로드시 사용할 임베딩 모델
    allow_dangerous_deserialization = True # 신뢰된 파일만 사용(pickle 역직렬화 허용)
)

In [21]:
search_result = vector_db.similarity_search(
    query='학교 선생님이 아끼는 해부학 책은 누가 찢었는가?', # 쿼리: 한글
    k = 4 # 상위 4개 Document
)

search_result # list[Document]

[Document(id='927e34ff-ba42-449c-b3d5-e0dc42f595fa', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 15, 'page_label': '16'}, page_content='talking about it. Becky wanted to talk to Tom, but he \ndidn’t look at her. \nThen Tom talked to Amy. Becky watched him and she \nwas angry. She said to her friends, “I’m going to have an \nadventure day. You can come on my adventure.” But she \ndidn’t ask Tom. \nLater in the morning, Tom ta lked to Amy again. Becky \ntalked to her friend Alfred and looked at a picture-book \nwith him. Tom watched them and he was angry with \nBecky. \nIn the afternoon, Tom waited for Becky at the school \nfence. He said, “I’m sorry.” \nBut Beck

In [22]:
for i, doc in enumerate(search_result, 1):
    # 사람이 보는 페이지 : 본문 내용
    print(f"{i}번째 {doc.metadata['page_label']} page: {doc.page_content}")

1번째 16 page: talking about it. Becky wanted to talk to Tom, but he 
didn’t look at her. 
Then Tom talked to Amy. Becky watched him and she 
was angry. She said to her friends, “I’m going to have an 
adventure day. You can come on my adventure.” But she 
didn’t ask Tom. 
Later in the morning, Tom ta lked to Amy again. Becky 
talked to her friend Alfred and looked at a picture-book 
with him. Tom watched them and he was angry with 
Becky. 
In the afternoon, Tom waited for Becky at the school 
fence. He said, “I’m sorry.” 
But Becky didn’t listen to him. She walked into the 
school room. The teacher’s new book was on his table. 
This book wasn’t for children, but Becky wanted to look 
at it. She opened the book quietly and looked at the 
pictures. 
Suddenly, Tom came into the room. Becky was 
surprised. She closed the book quickly, and it tore. Becky 
was angry with Tom and quickly went out of the room. 
Then the children and the teacher came into the room 
and went to their places. The t

### VectorStoreRetriever

리트리버는 벡터DB의 검색 기능을 표준화하고 추상화하여 LangChain 생태계에서 재사용성을 높이는 어댑터(Adapter) 역할을 수행한다.

벡터 저장소를 **`Retriever`라는 표준 인터페이스(Runnable)로 변환**한 뒤 실행하는 방식이다.

단순 유사도 검색뿐만 아니라, `search_type` 설정을 통해 **MMR(다양성 확보), 임계값 필터링(score_threshold)** 등 고급 검색 로직을 쉽게 적용할 수 있다.

**LCEL(LangChain Expression Language)** 파이프라인(`chain = retriever | llm`)에 즉시 통합 가능하다. 코드 수정 없이 검색 알고리즘만 교체하기 쉽다.

In [23]:
# 벡터 검색 (Retriever) 결과를 Context 에 넣고, PDF 기반 RAG 답변을 생성하는 코드
# VectorStore를 Retriever 인터페이스 변환
retriever = vector_db.as_retriever(
    search_type = 'similarity', # 검색 방식 : 코사인 유사도
    search_kwargs = {           # 검색 파라미터 묶음
        "k":3 # 상위3개
    }
)

# 질의 실행 -> list[Document]
search_results = retriever.invoke("마을 무덤의 남자는 누가 죽였는가?")

for i, doc in enumerate(search_results, 1): 
    # 사람이 보는 페이지 : 본문 내용
    print(f"{i}번째 {doc.metadata['page_label']} page : ")
    print(f"{doc.page_content} + \n")


1번째 19 page : 
A man asked him, “Where were you on the night of  
June 17th?” 
“I was in the graveyard,” Tom answered. 
“Did you see any people there?” the man asked: 
“Yes. Injun Joe, the doctor, and Muff Potter were there. 
They didn’t see me because I was behind some big trees.” 
“What did you see?” the man asked. 
“Injun Joe and the doctor talked angrily,” Tom 
answered. “Then Injun Joe killed the doctor with his knife. 
Muff Potter didn’t do it.” 
The people at the trial were surprised. Injun Joe quickly 
went out of the building. 
Tom and Huck were very afraid. Tom said, “Now Injun 
Joe knows about us. He can kill us, too.” 
Many people wanted to hear about the boys’ adventure 
in the graveyard. Tom liked talking about it. He was 
happy, too, because he helped Muff Potter. But he didn’t 
sleep well because he was afraid of Injun Joe. 
 
Chapter 7    Injun Joe’s Treasure 
 
One Saturday afternoon, Tom wanted to have an adventure 
because he didn’t want to think about Injun Joe. He

In [24]:
# 벡터 검색 (Retriever) 결과를 Context 에 넣고, PDF 기반 RAG 답변을 생성하는 코드
retriever = vector_db.as_retriever(
    search_type = 'similarity', # 검색 방식 : 코사인 유사도
    search_kwargs = {           # 검색 파라미터 묶음
        "k":3 # 상위3개
    }
)

prompt = PromptTemplate.from_template('''
    사용자의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.
    Context : {context}
    Question : {question}
''')

llm = init_chat_model('openai:gpt-5.6-luna')
output_parser = StrOutputParser() # 최종 출력은 텍스트 형식

# question 은 입력값 그대로 전달, context는 검색 + 문서 합치기 | 최종 프롬프트 완성 | LLM 호출 | 문자열 파싱
chain = (
    {'question' : RunnablePassthrough(), 'context': retriever | format_docs} | prompt | llm | output_parser
)

print(chain.invoke('마을 무덤의 남자는 누가 죽였는가?'))

마을 무덤에서 의사를 죽인 사람은 **인준 조(Injun Joe)**입니다.


# 음식 리뷰 조회 RAG

- 데이터셋 : fine_food_reviews_1k.csv
- 벡터DB 구성
- Retriever + llm 체인으로 리뷰 조회하는 기능

In [25]:
import pandas as pd

df = pd.read_csv('fine_food_reviews_1k.csv')
data = df['Text'].to_list()

data

['Wanted to save some to bring to my Chicago family but my North Carolina family ate all 4 boxes before I could pack. These are excellent...could serve to anyone',
 'Not pleased at all. When I opened the box, most of the rings were broken in pieces. A total waste of money.',
 'I\'m not sure that custard is really custard without eggs.  But this comes close.  I got it for use in a "Vegan pancake" recipe.  We were having houseguests who were Vegan and I wanted to make some special breakfasts while they were here.  One of the cooking/recipe sites had a recipe using this and there were lots of great reviews.  I tried the recipe and it turned out like wallpaper paste -- yuck!<br />However, the  so-called custard isn\'t so bad.  I think it\'s probably just cornstarch and annatto (yellow coloring with a slight flavor).  It\'s fun playing with it.  You could dress it up with fruit.  Seems to come out on the thin side when you make it as directed, so I use less milk because I like my custards t

In [26]:
vector_store = FAISS.from_texts(
    texts=data,
    embedding=embeddings,
    normalize_L2=True,  # 정규화 후 L2 검색 순위는 코사인 유사도 순위와 동일
)  # 리뷰 텍스트를 임베딩 -> FAISS 벡터 스토어 생성

# Retriever Chain 구성 (코사인 유사도, 상위 10개 리뷰 검색)
retriever = vector_store.as_retriever(
    search_type = 'similarity', # 검색 방식 : 코사인 유사도
    search_kwargs = {           # 검색 파라미터 묶음
        "k":10 
    }
)

prompt = PromptTemplate.from_template('''
    Context:
    {context}

    Question:
    {question}
''')

llm = init_chat_model('openai:gpt-5.6-luna')
output_parser = StrOutputParser() # 최종 출력은 텍스트 형식

# question 은 입력값 그대로 전달, context는 검색 + 문서 합치기 | 최종 프롬프트 완성 | LLM 호출 | 문자열 파싱
chain = (
    {'question' : RunnablePassthrough(), 'context': retriever | format_docs} | prompt | llm | output_parser
)

question = 'fresh fruit'
print(chain.invoke(question))


The fresh fruit review is for **rambutan**. The reviewer praised its soft, spiky exterior and juicy center, saying it arrived in good condition, lasted nearly a week refrigerated, and was a big hit with 4th graders.
